<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms---W7/blob/main/W7D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

main.py

In [ ]:
import os
import pandas as pd
import chromadb

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


PDF_PATH = "data/Lecture_1_Cyber_Security.pdf"
RESULTS_DIR = "results"
DATABASE_PATH = "chroma_db"

os.makedirs(RESULTS_DIR, exist_ok=True)

CONFIGURATIONS = {
    "small_180_30": {
        "chunk_size": 180,
        "overlap": 30
    },
    "medium_500_100": {
        "chunk_size": 500,
        "overlap": 100
    },
    "large_1200_200": {
        "chunk_size": 1200,
        "overlap": 200
    }
}


QUESTIONS = [
    {
        "id": "Q1",
        "question": "What is the objective of computer security and its three main properties?",
        "gold_pages": [6, 7]
    },
    {
        "id": "Q2",
        "question": "What are confidentiality and privacy, and how are they different?",
        "gold_pages": [7, 8]
    },
    {
        "id": "Q3",
        "question": "What are passive and active attacks? Give their types.",
        "gold_pages": [13, 14, 15]
    },
    {
        "id": "Q4",
        "question": "What is the difference between data integrity and system integrity?",
        "gold_pages": [9]
    },
    {
        "id": "Q5",
        "question": "What is a Security Mechanism?",
        "gold_pages": [11]
    },
    {
        "id": "Q6",
        "question": "Which specific encryption algorithm does the lecture recommend for confidentiality?",
        "gold_pages": []
    },
    {
        "id": "Q7",
        "question": "What are the four tasks involved in designing a security service?",
        "gold_pages": [17]
    },
    {
        "id": "Q8",
        "question": "What is cybersecurity, and what other terms are used for it?",
        "gold_pages": [19]
    },
    {
        "id": "Q9",
        "question": "What are the main cybersecurity categories and their core ideas?",
        "gold_pages": [21, 22, 23]
    },
    {
        "id": "Q10",
        "question": "In which year was the first official cybersecurity definition introduced according to the lecture?",
        "gold_pages": []
    }
]


def load_pdf(path):
    reader = PdfReader(path)
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if text.strip():
            pages.append({
                "page": page_number,
                "text": text
            })

    return pages


def create_chunks(pages, chunk_size, overlap):
    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size.")

    chunks = []
    step = chunk_size - overlap

    for page in pages:
        text = page["text"]

        for start in range(0, len(text), step):
            chunk_text = text[start:start + chunk_size]

            if not chunk_text.strip():
                continue

            chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": os.path.basename(PDF_PATH),
                    "page": page["page"],
                    "chunk_start_char": start,
                    "chunk_size": chunk_size,
                    "overlap": overlap
                }
            })

    return chunks


print("Loading embedding model...")

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

client = chromadb.PersistentClient(
    path=DATABASE_PATH
)

print("Loading PDF document...")

pages = load_pdf(PDF_PATH)

print("Extracted pages:", len(pages))

all_results = []
comparison_results = []

for config_name, settings in CONFIGURATIONS.items():

    print("\nRunning:", config_name)

    chunk_size = settings["chunk_size"]
    overlap = settings["overlap"]

    chunks = create_chunks(
        pages,
        chunk_size,
        overlap
    )

    print("Number of chunks:", len(chunks))

    documents = [chunk["text"] for chunk in chunks]

    embeddings = model.encode(
        documents,
        convert_to_numpy=True
    ).tolist()

    metadatas = [chunk["metadata"] for chunk in chunks]

    ids = [
        f"{config_name}_chunk_{i}"
        for i in range(len(chunks))
    ]

    try:
        client.delete_collection(config_name)
    except Exception:
        pass

    collection = client.create_collection(
        name=config_name,
        metadata={"hnsw:space": "cosine"}
    )

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    for item in QUESTIONS:

        query_embedding = model.encode(
            [item["question"]],
            convert_to_numpy=True
        ).tolist()

        retrieved = collection.query(
            query_embeddings=query_embedding,
            n_results=min(3, len(chunks)),
            include=[
                "documents",
                "metadatas",
                "distances"
            ]
        )

        retrieved_pages = []
        retrieved_ranks = []

        for rank, metadata in enumerate(
            retrieved["metadatas"][0],
            start=1
        ):
            page_number = metadata["page"]

            retrieved_pages.append(page_number)
            retrieved_ranks.append(rank)

            all_results.append({
                "configuration": config_name,
                "question_id": item["id"],
                "question": item["question"],
                "rank": rank,
                "retrieved_page": page_number,
                "distance": retrieved["distances"][0][rank - 1],
                "chunk": retrieved["documents"][0][rank - 1],
                "source": metadata["source"]
            })

        gold_pages = item["gold_pages"]

        if gold_pages:
            found_pages = set(retrieved_pages)

            complete_evidence = set(gold_pages).issubset(found_pages)

            matching_ranks = [
                rank
                for page, rank in zip(
                    retrieved_pages,
                    retrieved_ranks
                )
                if page in gold_pages
            ]

            best_rank = (
                min(matching_ranks)
                if matching_ranks
                else "Not Found"
            )

            status = (
                "Complete Evidence Retrieved"
                if complete_evidence
                else "Incomplete / Missing Evidence"
            )

        else:
            complete_evidence = "N/A"
            best_rank = "N/A"
            status = "Unanswerable Question"

        comparison_results.append({
            "question_id": item["id"],
            "configuration": config_name,
            "retrieved_pages": retrieved_pages,
            "expected_pages": gold_pages,
            "complete_evidence": complete_evidence,
            "best_matching_rank": best_rank,
            "status": status
        })


all_results_df = pd.DataFrame(all_results)
comparison_df = pd.DataFrame(comparison_results)

all_results_df.to_csv(
    os.path.join(RESULTS_DIR, "top3_retrieval_results.csv"),
    index=False
)

comparison_df.to_csv(
    os.path.join(RESULTS_DIR, "comparison_10x3.csv"),
    index=False
)

print("\nExperiment completed successfully!")

print("\nComparison Results:")
print(comparison_df.to_string(index=False))

print("\nGenerated files:")
print("- results/top3_retrieval_results.csv")
print("- results/comparison_10x3.csv")

 questions.json

In [ ]:
[
  {
    "id": "Q1",
    "question": "What is the objective of computer security and its three main properties?",
    "gold_pages": [6, 7]
  },
  {
    "id": "Q2",
    "question": "What are confidentiality and privacy, and how are they different?",
    "gold_pages": [7, 8]
  },
  {
    "id": "Q3",
    "question": "What are passive and active attacks? Give their types.",
    "gold_pages": [13, 14, 15]
  },
  {
    "id": "Q4",
    "question": "What is the difference between data integrity and system integrity?",
    "gold_pages": [9]
  },
  {
    "id": "Q5",
    "question": "What is a Security Mechanism?",
    "gold_pages": [11]
  },
  {
    "id": "Q6",
    "question": "Which specific encryption algorithm does the lecture recommend for confidentiality?",
    "gold_pages": []
  },
  {
    "id": "Q7",
    "question": "What are the four tasks involved in designing a security service?",
    "gold_pages": [17]
  },
  {
    "id": "Q8",
    "question": "What is cybersecurity, and what other terms are used for it?",
    "gold_pages": [19]
  },
  {
    "id": "Q9",
    "question": "What are the main cybersecurity categories and their core ideas?",
    "gold_pages": [21, 22, 23]
  },
  {
    "id": "Q10",
    "question": "In which year was the first official cybersecurity definition introduced according to the lecture?",
    "gold_pages": []
  }
]

README.md

تجربة قواعد البيانات المتجهية وRAG - مقارنة طرق تقسيم النصوص (Chunking Experiment)

1. وصف المشروع

يهدف هذا المشروع إلى تجربة ومقارنة تأثير أحجام تقسيم النصوص المختلفة (Chunk Sizes) والتداخل بين الأجزاء (Overlap) على أداء أنظمة الاسترجاع المعزز بالتوليد (Retrieval-Augmented Generation - RAG).

نستخدم ملف PDF حقيقي يحتوي على محاضرة عن مقدمة في الأمن السيبراني (Introduction to Cyber Security)، ونقسم محتواه إلى أجزاء بأحجام مختلفة، ثم نخزن هذه الأجزاء داخل قاعدة بيانات متجهية (Vector Database) باستخدام ChromaDB.

بعد ذلك، نختبر قدرة النظام على استرجاع المعلومات الصحيحة عند طرح مجموعة من الأسئلة على محتوى المحاضرة.

2. أهداف المشروع

- فهم مفهوم Vector Databases واستخدامها.
- فهم آلية عمل RAG.
- تجربة طرق مختلفة لتقسيم النصوص (Text Chunking).
- مقارنة تأثير حجم الجزء والتداخل بين الأجزاء على نتائج الاسترجاع.
- استخدام Embeddings لتحويل النصوص إلى تمثيلات رقمية.
- تقييم جودة استرجاع المعلومات باستخدام مجموعة من الأسئلة.

3. مجموعة البيانات (Dataset)

تم استخدام ملف PDF حقيقي بعنوان:

"Lecture_1_Cyber_Security.pdf"

- عدد الصفحات: 26 صفحة.
- الموضوع: مقدمة في الأمن السيبراني.
- المؤسسة التعليمية: Borg El Arab Technological University.
- يحتوي الملف على موضوعات مثل:
  - Computer Security
  - CIA Triad
  - Confidentiality
  - Integrity
  - Availability
  - Security Attacks
  - Passive and Active Attacks
  - Cybersecurity Categories

4. الأدوات والتقنيات المستخدمة

الأداة| الاستخدام
Python| تنفيذ المشروع
ChromaDB| تخزين واسترجاع النصوص المتجهية
Sentence Transformers| إنشاء النصوص المتجهية (Embeddings)
Pandas| تنظيم وتحليل النتائج
PyPDF| استخراج النصوص من ملفات PDF
Multilingual MiniLM| نموذج تحويل النصوص إلى Embeddings

نموذج Embedding المستخدم:

"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

5. تجارب تقسيم النصوص (Chunking Configurations)

تم اختيار ثلاث إعدادات مختلفة للمقارنة:

Configuration| Chunk Size| Overlap
Small| 180| 30
Medium| 500| 100
Large| 1200| 200

الهدف هو معرفة أي إعداد يحقق أفضل استرجاع للمعلومات المطلوبة.

6. الأسئلة المستخدمة في التقييم

تم إعداد 10 أسئلة لاختبار النظام:

- 8 أسئلة يمكن العثور على إجاباتها داخل المحاضرة.
- سؤالان لا توجد إجاباتهما داخل الملف.

بعض الأسئلة تحتاج إلى جمع المعلومات من أكثر من صفحة، وذلك لاختبار قدرة النظام على استرجاع المعلومات الموزعة عبر أجزاء مختلفة من المستند.

7. آلية تنفيذ المشروع

1. قراءة ملف PDF واستخراج النصوص منه.
2. تقسيم النصوص باستخدام إعدادات Chunking الثلاثة.
3. إنشاء Embeddings لكل جزء باستخدام نموذج Sentence Transformers.
4. تخزين الأجزاء والـ Embeddings داخل ChromaDB.
5. إضافة Metadata لكل جزء، تشمل:
   - اسم الملف (Source Filename).
   - رقم الصفحة (Page Number).
   - موضع الجزء داخل النص (Chunk Start Character).
   - حجم الجزء (Chunk Size).
   - مقدار التداخل (Overlap).
6. إدخال الأسئلة العشرة إلى النظام.
7. استرجاع أفضل 3 نتائج (Top 3) لكل سؤال ولكل إعداد.
8. مقارنة النتائج وتقييم مدى ظهور المعلومات المطلوبة ضمن النتائج المسترجعة.

8. هيكل المشروع (Project Structure)

W7D2_VectorDB_Project/
│
├── main.py
├── questions.json
├── requirements.txt
│
├── data/
│   └── Lecture_1_Cyber_Security.pdf
│
├── results/
│   ├── top3_retrieval_results.csv
│   └── comparison_10x3.csv
│
└── README.md

9. طريقة تشغيل المشروع

الخطوة الأولى: تثبيت المكتبات

pip install -r requirements.txt

الخطوة الثانية: التأكد من وجود ملف البيانات

يجب وضع ملف PDF داخل مجلد "data" بالاسم التالي:

"Lecture_1_Cyber_Security.pdf"

الخطوة الثالثة: تشغيل المشروع

python main.py

عند التشغيل، سيقوم البرنامج بإنشاء قاعدة البيانات وتنفيذ تجارب الاسترجاع وحفظ ملفات النتائج داخل مجلد "results".

10. النتائج والتقييم (Results & Evaluation)

سيتم تقييم كل إعداد من إعدادات Chunking الثلاثة من خلال:

- استرجاع أفضل 3 أجزاء لكل سؤال.
- التحقق من وجود الصفحات التي تحتوي على الإجابة الصحيحة.
- تسجيل ترتيب المعلومات المسترجعة.
- مقارنة أداء الإعدادات الثلاثة.

سيتم استخدام جدول مقارنة يحتوي على 10 أسئلة و3 إعدادات مختلفة.

ملاحظة: لم يتم تسجيل نتائج فعلية أو تحديد أفضل إعداد حتى الآن. سيتم تحديث هذا القسم بعد تشغيل التجربة وتحليل ملفات النتائج.

11. تحليل النتائج (Analysis)

بعد تنفيذ التجربة، سيتم تحديد:

- أفضل إعداد من حيث استرجاع المعلومات المطلوبة.
- أسوأ إعداد في عملية الاسترجاع.
- تأثير حجم الـ Chunk على دقة النتائج.
- تأثير الـ Overlap على الاحتفاظ بالسياق.
- أسباب فشل بعض عمليات الاسترجاع، إن وجدت.

12. الخلاصة (Conclusion)

تساعد هذه التجربة على فهم تأثير تقسيم النصوص على جودة استرجاع المعلومات في أنظمة RAG.

ومن خلال مقارنة ثلاثة أحجام مختلفة من Chunking، يمكن دراسة العلاقة بين حجم الجزء، والتداخل بين الأجزاء، وقدرة قاعدة البيانات المتجهية على استرجاع المعلومات الصحيحة من المستندات.

requirment.txt

In [ ]:
chromadb>=0.5.0
sentence-transformers>=3.0.0
pypdf>=4.0.0
pandas>=2.0.0